In [1]:
# CELL 1: THE ULTIMATE SAFE OFFLINE INSTALLER
import sys, subprocess, os, importlib, site

print("🧹 Installing Unsloth while PROTECTING Kaggle's native environment...")

wheel_files = []
# 🛡️ THE SHIELD: Do not let offline wheels overwrite Kaggle's native core!
toxic_packages = ["torch-", "torch==", "torchvision", "torchaudio", "numpy", "scipy", "pillow"]

for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f.endswith(".whl") or f.endswith(".tar.gz"):
            # If the file contains any of the toxic names, skip it.
            if any(toxic in f.lower() for toxic in toxic_packages):
                continue
            wheel_files.append(os.path.join(root, f))

print(f"📦 Installing {len(wheel_files)} offline AI packages safely...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-index", "--no-deps"] + wheel_files)

importlib.reload(site)
print("✅ Safe install complete. Native PyTorch and NumPy are perfectly intact!")

🧹 Installing Unsloth while PROTECTING Kaggle's native environment...
📦 Installing 98 offline AI packages safely...
✅ Safe install complete. Native PyTorch and NumPy are perfectly intact!


# CELL 2: THE DATASET (UPDATED FOR MASKING)

In [2]:
# CELL 2: THE DATASET (UPDATED FOR MASKING)
import pandas as pd
from datasets import Dataset
import os

print("🔍 Auto-discovering Competition Data...")

train_path = None
# Scan the entire Kaggle input folder to find where the CSV is hiding
for root, dirs, files in os.walk("/kaggle/input"):
    if "train.csv" in files:
        train_path = os.path.join(root, "train.csv")
        break 

if not train_path:
    raise FileNotFoundError("❌ CRITICAL: Could not find train.csv anywhere in /kaggle/input/")

print(f"✅ Found Dataset at: {train_path}")

# Load the raw data
train_df = pd.read_csv(train_path)
available_columns = train_df.columns.tolist()

# Smart-search for the Question column
q_candidates = ['question', 'problem', 'prompt', 'instruction', 'text']
q_col = next((col for col in available_columns if col.lower() in q_candidates), available_columns[0])

# Smart-search for the Answer column
a_candidates = ['answer', 'solution', 'response', 'output', 'target', 'expected_answer']
a_col = next((col for col in available_columns if col.lower() in a_candidates), available_columns[1] if len(available_columns) > 1 else available_columns[-1])

print("📊 Loading Power Run Data (2500 rows)...")
train_df = train_df.sample(n=min(2500, len(train_df)), random_state=42)

def format_train(example):
    question_text = example[q_col]
    answer_text = str(example[a_col]).strip()
    # 👇 CHANGED: Added '### Assistant:\n' as a strict separator for the loss mask
    return {"text": f"User: Solve this reasoning task step-by-step.\nTask: {question_text}\n\n### Assistant:\n\\boxed{{{answer_text}}}"}

train_dataset = Dataset.from_pandas(train_df).map(format_train)
print(f"✅ Prepared {len(train_dataset)} high-quality training examples with separators.")

🔍 Auto-discovering Competition Data...
✅ Found Dataset at: /kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv
📊 Loading Power Run Data (2500 rows)...


Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

✅ Prepared 2500 high-quality training examples with separators.


In [3]:
# CELL 3: AWAKEN THE BEAST 
import importlib.util

# Cloak torchvision to avoid the basic offline mismatch
real_find_spec = importlib.util.find_spec
def hooked_find_spec(name, package=None):
    if "torchvision" in name: return None
    return real_find_spec(name, package)
importlib.util.find_spec = hooked_find_spec

from unsloth import FastLanguageModel 
import torch

print("⏳ Waking up the model... (IGNORE ANY RED WARNINGS IF THE LOADING BAR IS MOVING!)")

MODEL_ID = "/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=2048, 
    dtype=torch.bfloat16, 
    load_in_4bit=True,    
    trust_remote_code=True
)

print("✅ SUCCESS! The Beast is awake.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
⏳ Waking up the model... (IGNORE ANY RED WARNINGS IF THE LOADING BAR IS MOVING!)
Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2026.3.10: Fast Nemotron_H patching. Transformers: 5.3.0.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Server Edition. Num GPUs = 1. Max memory: 94.971 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Nemotron_H does not support SDPA - switching to fast e

Loading weights:   0%|          | 0/6243 [00:00<?, ?it/s]

/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1 does not have a padding token! Will use pad_token = <SPECIAL_999>.
✅ SUCCESS! The Beast is awake.


In [5]:
# CELL 4: THE ULTIMATE NEMOTRON MOE FIX (Native PyTorch Masking) - RUN #4
import gc
import torch
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel

# 1. Memory Flush
gc.collect()
torch.cuda.empty_cache()

# 2. Native PEFT Attachment (Attention + MLPs Targeted)
if not isinstance(model, PeftModel):
    print("🧠 Prepping for 8-bit/4-bit Training...")
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    
    # Targeting ALL linear layers so it learns logic, not just routing
    lora_config = LoraConfig(
        r=16, 
        lora_alpha=32,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"], 
        lora_dropout=0.0, 
        bias="none", 
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_config)
    print("✅ LoRA Attached (Attention + MLPs Targeted).")

# 3. The "Force-Multiplier" DNA Patch (Kaggle Quantization Fix)
def apply_final_patch(model):
    for module in model.modules():
        if hasattr(module.__class__, "moe") and not hasattr(module.__class__, "_is_final_patched"):
            def fixed_moe(self, hidden_states, topk_indices, topk_weights):
                orig_shape = hidden_states.shape
                topk_weights = topk_weights.to(hidden_states.dtype)
                hidden_states = hidden_states.view(-1, hidden_states.shape[-1])
                final_hidden_states = torch.zeros_like(hidden_states)
                
                for expert_idx, expert in enumerate(self.experts):
                    token_indices = (topk_indices == expert_idx).nonzero(as_tuple=True)[0]
                    if token_indices.numel() > 0:
                        expert_input = hidden_states[token_indices]
                        expert_weights = topk_weights[token_indices, (topk_indices == expert_idx).nonzero(as_tuple=True)[1]]
                        expert_output = expert(expert_input)
                        final_hidden_states.index_add_(0, token_indices, (expert_output * expert_weights.unsqueeze(-1)).to(final_hidden_states.dtype))
                    else:
                        dummy_input = torch.zeros((1, hidden_states.shape[-1]), device=hidden_states.device, dtype=hidden_states.dtype)
                        dummy_out = expert(dummy_input)
                        final_hidden_states = final_hidden_states + (dummy_out * 0.0).sum()
                return final_hidden_states.view(*orig_shape)

            module.__class__.moe = fixed_moe
            module.__class__._is_final_patched = True
            print("✅ MoE Router DNA Re-written (Quantization-Safe).")
            break

apply_final_patch(model)

# 4. Final Dtype Alignment
for param in model.parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.bfloat16)

# 5. Native Tokenization AND -100 Masking (BYPASSING TRL COMPLETELY)
def tokenize_and_mask(example):
    response_template = "### Assistant:\n"
    
    # Split the string to isolate just the prompt part
    if response_template in example["text"]:
        prompt_with_template = example["text"].split(response_template)[0] + response_template
    else:
        prompt_with_template = example["text"]
        
    # Tokenize the full text and just the prompt
    tokenized_full = tokenizer(example["text"], truncation=True, max_length=2048)
    tokenized_prompt = tokenizer(prompt_with_template, truncation=True, max_length=2048)
    
    input_ids = tokenized_full["input_ids"]
    labels = input_ids.copy()
    
    # Find exactly how many tokens make up the prompt instructions
    prompt_len = len(tokenized_prompt["input_ids"])
    
    # MASKING: Overwrite the prompt tokens in the labels array with -100
    mask_len = min(prompt_len, len(labels))
    for i in range(mask_len):
        labels[i] = -100
        
    return {"input_ids": input_ids, "attention_mask": tokenized_full["attention_mask"], "labels": labels}

print("✂️ Tokenizing dataset and applying native PyTorch -100 masking...")
# We use the native map function, completely removing the need for SFTTrainer
tokenized_dataset = train_dataset.map(tokenize_and_mask, remove_columns=train_dataset.column_names)
print("✅ Masking complete!")

# 6. Launch Native Trainer
trainer = Trainer(
    model=model,
    train_dataset=tokenized_dataset,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, padding=True),
    args=TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        num_train_epochs=1.0,  # 👈 1 full epoch to guarantee convergence
        learning_rate=2e-5, 
        warmup_ratio=0.05,
        bf16=True,
        logging_steps=5,
        optim="adamw_8bit",
        output_dir="outputs",
        report_to="none",
    ),
)

print("🚀 IGNITION! Commencing Run #4 (Native Masking & 1 Full Epoch).")
trainer.train()

🧠 Prepping for 8-bit/4-bit Training...
✅ LoRA Attached (Attention + MLPs Targeted).
✅ MoE Router DNA Re-written (Quantization-Safe).
✂️ Tokenizing dataset and applying native PyTorch -100 masking...


Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


✅ Masking complete!
🚀 IGNITION! Commencing Run #4 (Native Masking & 1 Full Epoch).


Step,Training Loss
5,18.998515
10,17.974910
15,16.952394
20,15.173950
25,9.522934
30,9.752460
35,9.343480
40,8.770300
45,8.401109
50,6.403997


TrainOutput(global_step=313, training_loss=6.838969727293752, metrics={'train_runtime': 9771.1, 'train_samples_per_second': 0.256, 'train_steps_per_second': 0.032, 'total_flos': 6.75135104699113e+16, 'train_loss': 6.838969727293752, 'epoch': 1.0})

In [6]:
import os
import zipfile
from IPython.display import FileLink

# 1. Define where to save the adapter
output_dir = "nvidia_nemotron_lora_adapter"
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"✅ Model and Tokenizer saved to {output_dir}")

# 2. Create the submission.zip
# The competition requires only the adapter files in the zip
submission_zip = "submission.zip"
files_to_include = [
    "adapter_config.json",
    "adapter_model.safetensors",
    "README.md",
    "special_tokens_map.json",
    "tokenizer_config.json",
    "tokenizer.json"
]

with zipfile.ZipFile(submission_zip, 'w') as zipf:
    for file in files_to_include:
        file_path = os.path.join(output_dir, file)
        if os.path.exists(file_path):
            zipf.write(file_path, arcname=file)
            print(f"📦 Added {file} to {submission_zip}")

print(f"\n🚀 {submission_zip} is ready for submission!")

# 3. Generate a download link
FileLink(submission_zip)

✅ Model and Tokenizer saved to nvidia_nemotron_lora_adapter
📦 Added adapter_config.json to submission.zip
📦 Added adapter_model.safetensors to submission.zip
📦 Added README.md to submission.zip
📦 Added tokenizer_config.json to submission.zip
📦 Added tokenizer.json to submission.zip

🚀 submission.zip is ready for submission!


/kaggle/working/submission.zip